In [1]:
import json
import openai
# from pathlib import Path
from tqdm import tqdm
import os

# Set the API key securely
from dotenv import load_dotenv
load_dotenv()


False

In [3]:
# Setting file paths
fold_path = os.path.join(os.path.dirname(os.getcwd()), 'data', 'pqal_fold5')
train_data = json.load(open(os.path.join(fold_path, "train_set.json")))
dev_data = json.load(open(os.path.join(fold_path, "dev_set.json")))

print(f"# Train examples: {len(train_data)}")
print(f"# Dev examples: {len(dev_data)}")


# Train examples: 450
# Dev examples: 50


In [7]:
# Pick 3 diverse, clear examples
few_shot_examples = []
for i, (pmid, entry) in enumerate(train_data.items()):
    if entry.get("final_decision") in {"yes", "no", "maybe"}:
        few_shot_examples.append({
            "question": entry["QUESTION"],
            "context": "\n".join(entry["CONTEXTS"]),
            "answer": entry["final_decision"]
        })
    if len(few_shot_examples) == 3:
        break


In [9]:
def build_prompt(question, context, few_shot=[]):
    examples = ""
    for i, ex in enumerate(few_shot):
        examples += f"Example {i+1}:\n"
        examples += f"QUESTION: {ex['question']}\n"
        examples += f"ABSTRACT: {ex['context']}\n"
        examples += f"ANSWER: {ex['answer']}\n\n"

    current_q = f"""Now, given the examples above, answer the following:

QUESTION: {question}
ABSTRACT: {context}
ANSWER:"""

    return f"""You are a biomedical researcher trained to answer yes/no/maybe questions based on scientific abstracts.

These are labeled examples:

{examples}{current_q}
"""


In [ ]:
def ask_openai(prompt, model="gpt-4"):
    try:
        response = openai.ChatCompletion.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        print("Error:", e)
        return "maybe"
